# dplyr -- Technical Reference

R's primary data manipulation package. dplyr provides a consistent set of
verbs for the most common data manipulation tasks. Other tidyverse packages
(tidyr, stringr, lubridate) are loaded individually in the sections that
need them.

## Quick Index

| Pattern | When to use | Section |
| :--- | :--- | :--- |
| Data inspection | First look at any data frame | SS1 |
| Selection and filtering | Subset rows and columns | SS2 |
| NA handling | Missing values | SS3 |
| Data manipulation | Clean, transform, create columns | SS4 |
| Joins and merges | Combine data frames | SS5 |
| Aggregation and GroupBy | Summarize, reshape, pivot | SS6 |
| Window operations | Rank, shift, cumulative, rolling | SS7 |
| Date and time | Date arithmetic, extraction | SS8 |
| String operations | Text cleaning and extraction | SS9 |
| I/O | Read and write data | SS10 |
| Performance tips | Speed and memory optimization | SS11 |

---
## When to Use

| Signal | Function to reach for |
| :--- | :--- |
| First look at a data frame | `glimpse()`, `summary()`, `head()` |
| Filter rows by condition | `filter()` |
| Select columns | `select()` |
| Replace NAs | `tidyr::replace_na()` or `coalesce()` |
| Apply condition to create a column | `if_else()` (binary) or `case_when()` (multi) |
| Summarize with aggregation | `group_by() |> summarise()` |
| Apply aggregation but keep original shape | `group_by() |> mutate()` |
| Rank within a group | `group_by() |> mutate(dense_rank(...))` |
| Compare to previous / next row | `group_by() |> mutate(lag(x))` / `lead(x)` |
| Running total within a group | `group_by() |> mutate(cumsum(x))` |
| Pivot rows to columns | `tidyr::pivot_wider()` |
| Combine two data frames on a key | `*_join()` family |
| Find rows with no match | `anti_join()` |
| Moving average | `slider::slide_dbl(x, mean, .before = n-1)` |
| Parse date strings | `lubridate::ymd()`, `ymd_hms()`, etc. |
| Extract date parts | `lubridate::year()`, `month()`, `wday()` |
| String pattern match | `stringr::str_detect()` |

---
## Sample Data

Run this cell first. All sections below reuse these frames.

In [ ]:
library(dplyr)

# Small reusable sample frame used throughout this reference
df <- tibble::tibble(
  user_id   = c(1, 2, 3, 4, 5, 6),
  age       = c(34, 28, NA, 41, 19, 55),
  region    = c("US", "EU", "US", "EU", "US", "EU"),
  platform  = c("iOS", "Android", "Web", "iOS", "Android", "Web"),
  revenue   = c(120, 95, NA, 200, 15, 310),
  active    = c(TRUE, TRUE, FALSE, TRUE, FALSE, TRUE)
)

A <- tibble::tibble(id = c(1, 2, 3, 4), val_a = c("a", "b", "c", "d"))
B <- tibble::tibble(id = c(2, 3, 4, 5), val_b = c("x", "y", "z", "w"))

print(df)

---
## SS1 -- Data Inspection

Always run these first. They tell you shape, types, missing values,
and distributions before you touch the data.

In [ ]:
library(dplyr)

# Shape and preview
dim(df)
head(df, 3)
tail(df, 3)

# Types and null counts
glimpse(df)

# Summary statistics
summary(df)

# Null summary
tibble::tibble(
  column     = names(df),
  null_count = colSums(is.na(df)),
  null_rate  = round(colMeans(is.na(df)), 3)
) |> dplyr::filter(null_count > 0)

# Cardinality
n_distinct(df$region)
table(df$platform)

---
## SS2 -- Selection and Filtering

`select()` picks columns, `filter()` picks rows by condition,
`slice()` picks rows by position. Row and column selection are
separate verbs chained with the pipe.

In [ ]:
library(dplyr)

# Column selection
df |> select(user_id, region)
df |> select(-active)             # drop a column
df |> pull(revenue)               # single column as bare vector

# Row selection by position
df |> slice(1:3)
df |> slice_head(n = 2)
df |> slice_tail(n = 2)

# Filter rows by condition
df |> filter(age > 30)
df |> filter(age > 30 & region == "US")
df |> filter(platform %in% c("iOS", "Android"))
df |> filter(!is.na(revenue))

# Row + column together
df |> filter(revenue > 100) |> select(user_id, region, revenue)

**Common mistakes:**
- `filter()` silently drops rows where the condition evaluates to `NA` -- always
  use `is.na()` explicitly when NA rows matter
- `select(-c(col1, col2))` to drop multiple columns at once -- the minus applies
  per column not as a vector subtraction
- `pull()` returns a plain vector; `select()` always returns a data frame

---
## SS3 -- NA Handling

`NA` is R's null marker. Unlike pandas, R aggregations return `NA`
if any input is `NA` -- always pass `na.rm = TRUE`.

In [ ]:
library(dplyr)
library(tidyr)

# Detect
is.na(df$revenue)
sum(is.na(df$revenue))

# Fill with a value
df |> mutate(revenue = replace_na(revenue, 0))

# Coalesce -- first non-NA wins
df |> mutate(revenue = coalesce(revenue, 0))

# Fill with group mean
df |> group_by(region) |>
  mutate(revenue = replace_na(revenue, mean(revenue, na.rm = TRUE))) |>
  ungroup()

# Forward fill (sort first)
df |> arrange(user_id) |> fill(revenue, .direction = "down")

# Drop rows with NAs
df |> drop_na(revenue)
df |> drop_na()

# Aggregations -- na.rm = TRUE is required, unlike pandas which skips NaN by default
sum(df$revenue, na.rm = TRUE)
mean(df$revenue, na.rm = TRUE)

**Common mistakes:**
- Forgetting `na.rm = TRUE` -- `sum(c(1, 2, NA))` returns `NA`, not `3`
- `df$col == NA` never works -- always use `is.na()`
- Forward-filling without `arrange()` first -- fills in row order, not time order

---
## SS4 -- Data Manipulation

Column creation, renaming, type casting, and conditional logic
built around `mutate()`.

In [ ]:
library(dplyr)

# Add / overwrite columns
df |> mutate(rev_per_user = revenue / n())
df |> mutate(log_rev = log1p(revenue))

# Rename
df |> rename(user = user_id, income = revenue)

# Type casting
df |> mutate(age = as.integer(age))
df |> mutate(platform = as.factor(platform))

# Conditional logic -- binary
df |> mutate(tier = if_else(revenue > 100, "High", "Low"))

# Conditional logic -- multi-branch (case_when, first match wins)
df |> mutate(tier = case_when(
  revenue > 200              ~ "Premium",
  revenue > 100              ~ "Standard",
  revenue > 0                ~ "Low",
  .default = "Unknown"
))

# Value lookup (discrete replacement)
df |> mutate(region_full = case_match(region,
  "US" ~ "United States",
  "EU" ~ "Europe",
  .default = region
))

# Binning
df |> mutate(age_group = cut(
  age,
  breaks = c(0, 18, 35, 55, 100),
  labels = c("Under 18", "18-34", "35-54", "55+"),
  include.lowest = TRUE
))

# Drop columns or rows
df |> select(-active)
df |> slice(-1)

**Common mistakes:**
- `if_else()` is strict about types -- both branches must return the same type
- `case_when(.default = col)` preserves unmatched values; without `.default` they
  become `NA`
- `rename(new = old)` -- new name comes first, opposite of a Python dict

---
## SS5 -- Joins and Merges

The `*_join()` family always matches on columns, never on row index --
no `.join()` vs `.merge()` ambiguity.

In [ ]:
library(dplyr)

# Basic joins
inner_join(A, B, by = "id")
left_join(A, B, by = "id")
right_join(A, B, by = "id")
full_join(A, B, by = "id")

# Join on differently named columns
left_join(A, B, by = c("id" = "id"))

# Handle overlapping column names
left_join(A, B, by = "id", suffix = c("_a", "_b"))

# Anti-join -- rows in A with no match in B
anti_join(A, B, by = "id")

# Stack vertically
bind_rows(A |> select(id), B |> select(id))

# Guard against fan-out (dplyr 1.1+)
tryCatch(
  left_join(A, B, by = "id", relationship = "one-to-one"),
  error = function(e) cat("cardinality error caught\n")
)

In [ ]:
# Demo
cat("inner_join:\n"); print(inner_join(A, B, by = "id"))
cat("\nanti_join (in A, not in B):\n"); print(anti_join(A, B, by = "id"))

**Common mistakes:**
- Joining on columns with different types -- dplyr raises a type-mismatch error,
  cast both keys to the same type first
- Forgetting `suffix =` when both frames share non-key column names -- dplyr adds
  `.x` / `.y` by default
- Row count grows after a left join -- check key uniqueness in the right table
  with `count(B, id)` before joining

---
## SS6 -- Aggregation and GroupBy

In [ ]:
library(dplyr)
library(tidyr)

# Basic aggregation
df |> summarise(
  total_rev    = sum(revenue, na.rm = TRUE),
  avg_rev      = mean(revenue, na.rm = TRUE),
  unique_users = n_distinct(user_id),
  n_rows       = n()
)

# Group by
df |> group_by(region) |> summarise(
  total_rev    = sum(revenue, na.rm = TRUE),
  avg_rev      = mean(revenue, na.rm = TRUE),
  unique_users = n_distinct(user_id),
  .groups = "drop"
)

# Conditional aggregation
df |> group_by(region) |> summarise(
  ios_rev = sum(if_else(platform == "iOS", revenue, 0), na.rm = TRUE),
  .groups = "drop"
)

# Group stats back on every row (mutate, not summarise)
df |> group_by(region) |> mutate(
  region_total = sum(revenue, na.rm = TRUE),
  pct_of_region = revenue / region_total
) |> ungroup()

# Pivot wider -- two categorical dimensions
df |> group_by(region, platform) |>
  summarise(revenue = sum(revenue, na.rm = TRUE), .groups = "drop") |>
  pivot_wider(names_from = platform, values_from = revenue, values_fill = 0)

**Common mistakes:**
- Forgetting `.groups = "drop"` in `summarise()` -- leaves result grouped and
  silently corrupts downstream operations
- Forgetting `ungroup()` after `group_by() |> mutate()`
- `pivot_wider()` without `values_fill = 0` -- missing combinations become `NA`

---
## SS7 -- Window Operations

In [ ]:
library(dplyr)
library(slider)

# Ranking
df |> mutate(
  row_num   = row_number(desc(revenue)),
  rnk       = min_rank(desc(revenue)),
  dense_rnk = dense_rank(desc(revenue))
)

# Rank within group
df |> group_by(region) |>
  mutate(rank_in_region = dense_rank(desc(revenue))) |>
  ungroup()

# Lag / lead (sort first)
df |> arrange(user_id) |> group_by(region) |> mutate(
  prev_rev = lag(revenue, 1),
  next_rev = lead(revenue, 1)
) |> ungroup()

# Running totals
df |> arrange(user_id) |> group_by(region) |> mutate(
  run_total = cumsum(replace_na(revenue, 0))
) |> ungroup()

# Rolling window (slider package)
df |> arrange(user_id) |> group_by(region) |> mutate(
  roll_mean_2 = slide_dbl(revenue, mean, .before = 1, na.rm = TRUE, .complete = FALSE)
) |> ungroup()

**Common mistakes:**
- Forgetting `arrange()` before `lag()` / `lead()` -- operates on row order, not
  time order
- Forgetting `ungroup()` after grouped `mutate()`
- `slide_dbl()` without wrapping in `lag(x, 1)` first includes the current row in
  rolling stats -- leakage risk in time-series features

---
## SS8 -- Date and Time

`lubridate` parses and manipulates dates. Functions are named after
the expected format: `ymd()`, `mdy()`, `ymd_hms()`, etc.

In [ ]:
library(dplyr)
library(lubridate)

dates_df <- tibble::tibble(
  user_id   = 1:4,
  event_ts  = c("2024-01-05 10:30:00", "2024-02-14 08:00:00",
                "2024-03-01 22:15:00", "2024-03-31 09:45:00"),
  signup    = as.Date(c("2023-06-01","2023-09-15","2024-01-01","2023-12-01"))
)

dates_df <- dates_df |> mutate(
  event_ts    = ymd_hms(event_ts),
  year        = year(event_ts),
  month       = month(event_ts),
  day         = day(event_ts),
  hour        = hour(event_ts),
  day_of_week = wday(event_ts, week_start = 1) - 1,   # 0=Mon, 6=Sun
  is_weekend  = as.integer(wday(event_ts, week_start = 1) - 1 >= 5),
  # Elapsed time
  days_since_signup = as.numeric(as_date(event_ts) - signup, units = "days"),
  # Date arithmetic
  plus_7d     = as_date(event_ts) + days(7),
  minus_1mo   = as_date(event_ts) %m-% months(1)   # calendar-aware month subtraction
)

print(dates_df |> select(user_id, event_ts, day_of_week, is_weekend, days_since_signup))

**Common mistakes:**
- `wday()` default is 1 = Sunday -- always pass `week_start = 1` and subtract 1
  to match Monday-origin conventions
- Plain `+ months(1)` on Jan 31 overflows into March -- use `%m+%` / `%m-%` for
  calendar-correct month arithmetic
- Using `.dt.*`-style chaining from pandas habit -- lubridate uses standalone
  functions like `year(x)`, `month(x)`, not `x$year()`

---
## SS9 -- String Operations

`stringr` provides vectorized string functions, all prefixed `str_*`.
Regex is always on -- use `fixed()` or `coll()` for literal matching.

In [ ]:
library(dplyr)
library(stringr)

str_df <- tibble::tibble(
  name    = c("  Alice Smith ", "BOB JONES", "carol white"),
  email   = c("alice@example.com", "bob@test.org", "carol@example.com"),
  revenue = c("$1,200", "$950", "$2,000")
)

str_df <- str_df |> mutate(
  name_clean    = str_to_title(str_trim(name)),
  domain        = str_extract(email, "@(.+)$"),
  is_example    = str_detect(email, "example\\.com"),
  revenue_clean = as.numeric(str_remove_all(revenue, "[$,]"))
)

print(str_df)

In [ ]:
library(stringr)

x <- c("hello world", "foo bar baz", "test_string")

# Case and whitespace
str_to_lower(x)
str_to_upper(x)
str_trim("  hello  ")

# Detect / match
str_detect(x, "bar")           # returns logical vector
str_starts(x, "foo")
str_ends(x, "baz")

# Replace
str_replace(x, "o", "0")      # first match only
str_replace_all(x, "o", "0")  # all matches (pandas default)
str_remove_all(x, "\\s+")      # remove all whitespace

# Extract
str_extract(x, "\\w+")        # first word
str_length(x)                  # character count
str_sub(x, 1, 3)               # first 3 characters (1-indexed, inclusive)

# Split
str_split("a,b,c", ",")       # returns a list

**Common mistakes:**
- `str_replace()` replaces the first match only -- use `str_replace_all()` for
  pandas' default all-occurrences behavior
- Regex is always on in stringr -- use `fixed(".")` when you mean a literal dot,
  not the regex wildcard
- `str_detect()` returns `NA` for `NA` input (not `FALSE`) -- `filter()` drops
  `NA` rows automatically, but `mutate()` keeps them as `NA`

---
## SS10 -- I/O

In [ ]:
library(readr)

# Read CSV
# df <- read_csv("file.csv")
# df <- read_csv("file.csv", col_select = c(user_id, date, revenue))
# df <- read_csv("file.csv", col_types = cols(user_id = col_character()))
# df <- read_csv("file.csv", n_max = 1000)

# Write
# write_csv(df, "output.csv")    # no row-names column (unlike base write.csv)

# JSON
# jsonlite::fromJSON("file.json")
# jsonlite::write_json(df, "output.json")

# Excel
# readxl::read_excel("file.xlsx", sheet = "Sheet1")
# writexl::write_xlsx(df, "output.xlsx")

# SQL
# conn <- DBI::dbConnect(RSQLite::SQLite(), "db.sqlite")
# df   <- DBI::dbGetQuery(conn, "SELECT * FROM table")

cat("read_csv() key arguments:\n")
cat("  col_select  : load only specific columns\n")
cat("  col_types   : specify types on load to avoid silent coercion\n")
cat("  n_max       : load first N rows only\n")
cat("  na          : strings to treat as NA (default: c('', 'NA'))\n")

**Common mistakes:**
- Using base `write.csv()` -- writes row names as an unnamed first column by
  default; use `readr::write_csv()` instead
- Not specifying `col_types` for ID columns -- numeric IDs load as `double` and
  lose leading zeros (zip codes, product codes)

---
## SS11 -- Performance Tips

In [ ]:
library(dplyr)

# Vectorized operations beat rowwise() every time
df |> mutate(rev2 = revenue * 2)                        # fast - vectorized
# df |> rowwise() |> mutate(rev2 = revenue * 2) |> ungroup()  # slow - row loop

# filter() early to shrink the frame before expensive operations
df |> filter(!is.na(revenue)) |> group_by(region) |> summarise(n = n(), .groups = "drop")

# Factor for low-cardinality string columns
df |> mutate(platform = as.factor(platform))

# Read only needed columns
# read_csv("large.csv", col_select = c(user_id, date, revenue))

# For large data: dtplyr (data.table backend), arrow, or duckdb
# dtplyr::lazy_dt(df) |> filter(...) |> as_tibble()

cat("Key performance rules:\n")
cat("  1. Avoid rowwise() -- replace with vectorized mutate()\n")
cat("  2. filter() early to reduce rows before grouping\n")
cat("  3. Use as.factor() for repeated low-cardinality strings\n")
cat("  4. col_select= when loading -- only read what you need\n")
cat("  5. dtplyr / arrow / duckdb for data that exceeds memory\n")

---
## Decision Guide

```
Aggregating data?
+-- One metric, one grouping dimension      -> group_by() |> summarise()
+-- Two categorical dimensions              -> summarise() |> pivot_wider()
+-- Frequency count                         -> count() / table()
+-- Stats back on every row                 -> group_by() |> mutate()

Filtering rows?
+-- Any condition                           -> filter()
+-- By position                             -> slice() / slice_head() / slice_tail()

Selecting columns?
+-- Keep specific columns                   -> select(col1, col2)
+-- Drop specific columns                   -> select(-col1)
+-- Single column as vector                 -> pull(col)

Labeling or bucketing?
+-- Binary condition                        -> if_else()
+-- Multiple conditions                     -> case_when(.default = val)
+-- Discrete value lookup                   -> case_match()
+-- Continuous bins                         -> cut(breaks = ...)

Combining data frames?
+-- Match on a key column                   -> *_join()
+-- Rows in A with no match in B            -> anti_join()
+-- Stack rows vertically                   -> bind_rows()

Window operations?
+-- Rank within group                       -> group_by() |> mutate(dense_rank())
+-- Previous / next value                   -> group_by() |> mutate(lag() / lead())
+-- Running total                           -> group_by() |> mutate(cumsum())
+-- Moving average                          -> slide_dbl() from slider package

NA handling?
+-- Fill with value                         -> replace_na() / coalesce()
+-- Fill from previous row                  -> arrange() then fill(.direction="down")
+-- Drop rows                               -> drop_na()
+-- Aggregate safely                        -> always pass na.rm = TRUE

Slow code?
+-- rowwise() in a mutate                   -> replace with vectorized mutate()
+-- Loading a large file fully              -> col_select= or n_max=
+-- Low-cardinality string column           -> as.factor()
```